# UABR-SLM · Notebook 11 — Post-hoc analyses, no training: (1) readout vs representation, (2) weight interpolation / task-vector re-addition, (6) threshold recalibration

GPU **T4** (×1) · ~30–40 min · validation only (test sealed). Uses the checkpoints of notebooks 06, 07 and 10.

**Why:** SFT damaged both decision tokens (07: routing AUROC 0.838 → 0.70; 10: IsRel 0.87 → 0.76). Is the information
gone from the model, or did only the readout break? Can damage be undone without replay?

| Analysis | What is measured | Reading rule (fixed before results) |
|---|---|---|
| **1 Readout vs representation** | Same post-RMSNorm hidden state h at the decision slot. The token score z_CALL − z_ANSWER = h·(e_CALL − e_ANSWER) is a *fixed* linear readout (tied embeddings). Compare it with (a) a linear probe **re-fitted** on the post-SFT h (half-B labels → validation), (b) the **pre-SFT probe** applied unchanged, (c) a label-budget curve (50 … all labels). Same for IsRel (critique slot; half-B windows → G2 / validation windows) | recovered share = (probe_post − token_post) / (token_pre − token_post): ≥ 0.8 **readout-dominated**, ≤ 0.2 **representation-dominated**, else **mixed**; paired clustered bootstrap CIs |
| **2 Weight interpolation / task vector** | θ_α = θ_route + α(θ_SFT − θ_route), α = 0 … 1 (step 0.1), per seed: routing AUROC, default call rate, instruction val loss, closed / evidence EM. Task vector: θ_SFT + β(θ_route − θ0), β ∈ {0.5, 1}. Same interpolation for the critic (IsRel) | SFT gain kept g(α) = (L_0 − L_α)/(L_0 − L_1). Report α* = largest α with routing AUROC ≥ E0 − 0.02 and its g; and the AUROC at the smallest α with g ≥ 0.9 |
| **6 Threshold recalibration** | Re-fit only the decision threshold (max balanced accuracy) on n ∈ {50, 100, 200, 500} labelled half-B questions (5 draws); compare with the default threshold 0 and with the re-fitted probe (same labels) | balanced accuracy vs the frozen labels, agreement with E0 decisions, call rate, utility with θ0 rewards (λ from 06) |

**Inputs:** outputs of notebooks **01, 02, 05, 06 (v2), 07, 10**.

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from uabr.data import tokenizer as T
from uabr.data import qa_format as Q
from uabr.eval import prompts as PR
from uabr.eval.batched import greedy_batch
from uabr.eval.generate import load_model
from uabr.eval.metrics import alias_em
from uabr.retrieval.select import select_non_overlapping
from uabr.routing.metrics import auroc, clustered_bootstrap_auroc
from uabr.routing.pilot import paired_cluster_bootstrap
from uabr.routing.router import router_prompt, router_scores
from uabr.critic.selfrag import critique_scores, last_hidden
from uabr.critic.windows import window_items
from uabr.analysis.readout import fit_probe, probe_scores, learning_curve, interpolate, task_vector_add, fit_threshold, balanced_accuracy
from uabr.train.sft import sft_val_loss

## 2 · Configuration and inputs

In [ ]:
import re
import numpy as np, pandas as pd
OUT = "/kaggle/working/11_posthoc" if ON_KAGGLE else os.path.abspath("outputs/11_posthoc")
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if ON_KAGGLE: assert DEV == "cuda", "No GPU - set Accelerator to GPU T4"
SEEDS = [int(s) for s in os.environ.get("SEEDS", "11,29").split(",")]
N_BOOT = int(os.environ.get("N_BOOT", 1000)); SEED = 1111
ALPHAS = [round(a, 1) for a in np.arange(0, 1.01, 0.1)]; C_ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]; BETAS = [0.5, 1.0]
NS = [50, 100, 200, 500, 1000, 100000]; REPS = 5; CAL_NS = [50, 100, 200, 500]
PRES_N = int(os.environ.get("PRES_N", 500)); MAX_NEW, SEQ = 16, 512

def one_input(pattern, env=None):
    if env and os.environ.get(env): return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert len(hits) == 1, f"{pattern}: expected exactly 1 attached file, found {len(hits)}: {hits}"
    return hits[0]

tok = T.load(find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER"))
top = pd.read_parquet(find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20")).set_index("id")
wins = pd.read_parquet(find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS"))
THETA0 = one_input("05_mixed/arm_M/model_final.pt", "THETA0")
POLICY_PATH = one_input("06_routing/routing_policy.json", "POLICY"); D06 = os.path.dirname(POLICY_PATH)
POLICY = json.load(open(POLICY_PATH)); LAM = float(POLICY["lambda"]); ROUTER = os.path.join(D06, "router_action.pt")
V = pd.read_parquet(os.path.join(D06, "labels_val.parquet")).reset_index(drop=True)
B = pd.read_parquet(os.path.join(D06, "labels_halfB.parquet")).reset_index(drop=True)
D07 = os.path.dirname(one_input("07_sft_pilot/pilot_report.json", "REPORT07"))
D10 = os.path.dirname(one_input("10_selfrag/critic.pt", "CRITIC"))
CRITIC = os.path.join(D10, "critic.pt")
def ckpts(d, seed):
    s = json.load(open(os.path.join(d, f"sft_s{seed}", "summary.json")))
    out = [(int(re.findall(r"\d+", os.path.basename(p))[0]) / s["updates"], p) for p in glob.glob(os.path.join(d, f"sft_s{seed}", "step_*.pt"))]
    return sorted(out) + [(1.0, os.path.join(d, f"sft_s{seed}", "final.pt"))]

E1 = {s: ckpts(D07, s) for s in SEEDS}; C1 = {s: os.path.join(D10, f"sft_s{s}", "final.pt") for s in SEEDS}
for s in SEEDS: assert os.path.exists(C1[s]) and all(os.path.exists(p) for _, p in E1[s]), s
WTEXT = dict(zip(wins["doc_id"], wins["text"]))
WART, WSTART = dict(zip(wins["doc_id"], wins["article_idx"])), dict(zip(wins["doc_id"], wins["start_word"]))
stop_ids = [i for t, i in tok.get_vocab().items() if "\n" in tok.decode([i])]
sx = pd.read_parquet(os.path.join(D07, "sft_examples.parquet"))
mk = lambda r: (list(map(int, r.tokens)), [False] * int(r.n_prompt) + [True] * (len(r.tokens) - int(r.n_prompt)))
SFT_VALX = [mk(r) for r in sx[sx.part == "val"].itertuples(index=False)]
PRESQ = V.sample(min(PRES_N, len(V)), random_state=SEED)
print(f"router {ROUTER} | theta0 {THETA0} | critic {CRITIC} | val {len(V):,} | half-B {len(B):,} | 07 checkpoints {[[round(f, 2) for f, _ in E1[s]] for s in SEEDS]}")

## 3 · Analysis 1a — routing: fixed token readout vs re-fitted probe vs frozen pre-SFT probe

In [ ]:
RP_B = [router_prompt(tok, q, "action") for q in B.question]; RP_V = [router_prompt(tok, q, "action") for q in V.question]
yB, yV, gV = B.y.astype(int).values, V.y.astype(int).values, V.group_id.values
def feats(path, pb, pv):
    m = load_model(path, DEV); hb, hv = last_hidden(m, pb, DEV), last_hidden(m, pv, DEV)
    sv = router_scores(m, pv, "action", DEV)[0]; del m
    if DEV == "cuda": torch.cuda.empty_cache()
    return hb, hv, sv
F = {("E0", "all", 0.0): feats(ROUTER, RP_B, RP_V)}
for s in SEEDS:
    for frac, p in E1[s]: F[("E1", s, frac)] = feats(p, RP_B, RP_V)
P0 = fit_probe(F[("E0", "all", 0.0)][0], yB)
R1 = []; SC = {}
for key, (hb, hv, sv) in F.items():
    pr = fit_probe(hb, yB)
    SC[key] = {"token": sv, "refit": probe_scores(pr, hv), "frozen": probe_scores(P0, hv)}
    R1.append({"model": key[0], "seed": key[1], "frac": key[2], "token": auroc(yV, sv), "refit_probe": auroc(yV, SC[key]["refit"]),
               "frozen_pre_probe": auroc(yV, SC[key]["frozen"])})
R1 = pd.DataFrame(R1); display(R1.round(4))
tok_pre = R1.loc[R1.model == "E0", "token"].item()
REC = []
for s in SEEDS:
    k1, k0 = ("E1", s, 1.0), ("E0", "all", 0.0)
    f = lambda idx, w, a="refit", b="token": auroc(yV[idx], SC[k1][a][idx]) if w == "a" else auroc(yV[idx], SC[k1][b][idx])
    d_probe_tok = paired_cluster_bootstrap(f, gV, n_boot=N_BOOT, seed=s)
    g = lambda idx, w: auroc(yV[idx], SC[k1]["refit"][idx]) if w == "a" else auroc(yV[idx], SC[k0]["token"][idx])
    d_probe_pre = paired_cluster_bootstrap(g, gV, n_boot=N_BOOT, seed=s)
    r = R1[(R1.seed == s) & (R1.frac == 1.0)].iloc[0]
    share = (r.refit_probe - r.token) / (tok_pre - r.token) if tok_pre - r.token >= 0.02 else np.nan
    REC.append({"token": "routing", "seed": s, "token_pre": tok_pre, "token_post": r.token, "refit_post": r.refit_probe,
                "refit_pre": R1.loc[R1.model == "E0", "refit_probe"].item(),
                "recovered_share": share, "refit_minus_token_post": d_probe_tok["diff"], "lo": d_probe_tok["ci_lo"], "hi": d_probe_tok["ci_hi"],
                "refit_post_minus_token_pre": d_probe_pre["diff"], "lo_pre": d_probe_pre["ci_lo"], "hi_pre": d_probe_pre["ci_hi"],
                "reading": "n/a (no damage)" if share != share else ("readout-dominated" if share >= 0.8 else ("representation-dominated" if share <= 0.2 else "mixed"))})
display(pd.DataFrame(REC).round(4))
LC = []
for key in [("E0", "all", 0.0)] + [("E1", s, 1.0) for s in SEEDS]:
    hb, hv, _ = F[key]
    LC.append(learning_curve(hb, yB, hv, yV, NS, reps=REPS, seed=SEED).assign(model=f"{key[0]}_s{key[1]}"))
LC = pd.concat(LC); print("probe AUROC on validation vs number of half-B labels (mean over draws):")
display(LC.groupby(["model", "n_labels"]).auroc.mean().unstack(0).round(4))

## 4 · Analysis 1b — IsRel (critique slot): token vs re-fitted probe vs frozen pre-SFT probe

In [ ]:
Bw = window_items(B, top, tok, WTEXT, WART, WSTART, 10, SEQ - MAX_NEW)
pos, neg = Bw[Bw.auto_rel == 1], Bw[Bw.auto_rel == 0]
hard = neg[neg.answer_in ^ neg.subject_in]; easy = neg.drop(hard.index); n_hard = min(len(hard), 3 * len(pos) // 2)
BW = pd.concat([pos, hard.sample(n_hard, random_state=1010), easy.sample(min(len(easy), 3 * len(pos) - n_hard), random_state=1010)])
def rebuild(df):
    df = df.copy(); df["prompt"] = [Q.evidence_prompt_ids(tok, q, [WTEXT[d]], SEQ - MAX_NEW)[0] for q, d in zip(df.question, df.doc_id)]
    return df
G2W = rebuild(pd.read_parquet(os.path.join(D10, "g2_windows.parquet"))); EVW = rebuild(pd.read_parquet(os.path.join(D10, "eval_windows.parquet")))
print(f"IsRel probe training windows {len(BW):,} (relevant {BW.auto_rel.mean():.1%}) | G2 {len(G2W)} | validation windows {len(EVW):,}")
def cfeats(path):
    m = load_model(path, DEV)
    out = {k: last_hidden(m, d.prompt.tolist(), DEV) for k, d in [("B", BW), ("G2", G2W), ("EV", EVW)]}
    out.update({f"tok_{k}": critique_scores(m, d.prompt.tolist(), DEV) for k, d in [("G2", G2W), ("EV", EVW)]})
    del m
    if DEV == "cuda": torch.cuda.empty_cache()
    return out
CF = {("critic", "all"): cfeats(CRITIC)}
for s in SEEDS: CF[("critic_SFT", s)] = cfeats(C1[s])
CP0 = fit_probe(CF[("critic", "all")]["B"], BW.auto_rel)
R2 = []; CSC = {}
for key, f in CF.items():
    pr = fit_probe(f["B"], BW.auto_rel)
    for ev, d, lab in [("G2", G2W, "g2_label"), ("EV", EVW, "auto_rel")]:
        CSC[(key, ev)] = {"token": f[f"tok_{ev}"], "refit": probe_scores(pr, f[ev]), "frozen": probe_scores(CP0, f[ev])}
        R2.append({"model": key[0], "seed": key[1], "eval": ev, "token": auroc(d[lab], f[f"tok_{ev}"]),
                   "refit_probe": auroc(d[lab], CSC[(key, ev)]["refit"]), "frozen_pre_probe": auroc(d[lab], CSC[(key, ev)]["frozen"])})
R2 = pd.DataFrame(R2); display(R2.round(4))
for ev, d, lab in [("G2", G2W, "g2_label"), ("EV", EVW, "auto_rel")]:
    pre = R2[(R2.model == "critic") & (R2["eval"] == ev)].token.item()
    for s in SEEDS:
        k1 = (("critic_SFT", s), ev)
        f = lambda idx, w: auroc(d[lab].values[idx], CSC[k1]["refit" if w == "a" else "token"][idx])
        r = paired_cluster_bootstrap(f, d.group_id.values, n_boot=N_BOOT, seed=s)
        row = R2[(R2.model == "critic_SFT") & (R2.seed == s) & (R2["eval"] == ev)].iloc[0]
        share = (row.refit_probe - row.token) / (pre - row.token) if pre - row.token >= 0.02 else np.nan
        REC.append({"token": f"IsRel_{ev}", "seed": s, "token_pre": pre, "token_post": row.token, "refit_post": row.refit_probe,
                    "refit_pre": R2[(R2.model == "critic") & (R2["eval"] == ev)].refit_probe.item(),
                    "recovered_share": share, "refit_minus_token_post": r["diff"], "lo": r["ci_lo"], "hi": r["ci_hi"],
                    "reading": "n/a (no damage)" if share != share else ("readout-dominated" if share >= 0.8 else ("representation-dominated" if share <= 0.2 else "mixed"))})
REC = pd.DataFrame(REC); REC["probe_drop_pre_to_post"] = REC.refit_pre - REC.refit_post; REC.to_csv(f"{OUT}/readout_recovery.csv", index=False)
print("Recovered share of the SFT damage by re-fitting a linear readout (rule: >=0.8 readout, <=0.2 representation):")
display(REC[["token", "seed", "token_pre", "token_post", "refit_pre", "refit_post", "probe_drop_pre_to_post", "recovered_share", "refit_minus_token_post", "lo", "hi", "reading"]].round(4))
print("probe_drop_pre_to_post = information lost from the representation as seen by a re-fitted linear readout (secondary)")

## 5 · Analysis 2 — weight interpolation θ_route → θ_SFT and task-vector re-addition

In [ ]:
CPq = [Q.closed_prompt_ids(tok, q) for q in PRESQ.question]
EPq = []
for r in PRESQ.itertuples(index=False):
    sel, _ = select_non_overlapping(list(top.loc[r.id].doc_ids), WART, WSTART, k=3, window_words=60)
    EPq.append(Q.evidence_prompt_ids(tok, r.question, [WTEXT[d] for d in sel], SEQ - MAX_NEW)[0])
def em(m, prompts, bans):
    g = greedy_batch(m, prompts, DEV, MAX_NEW, banned=bans, stop_ids=stop_ids)
    return float(np.mean([alias_em(PR.clean_generation(tok.decode(x)), a) for x, a in zip(g, PRESQ.gold_aliases)]))
def measure(m, with_em=True, critic=False):
    s = router_scores(m, RP_V, "action", DEV)[0]
    r = {"route_auroc": auroc(yV, s), "default_call": float((s > 0).mean()), "instr_val_loss": sft_val_loss(m, SFT_VALX, DEV)}
    bans = (T.CALL_ID, m.cfg.vocab_size - 2, m.cfg.vocab_size - 1) if critic else (T.CALL_ID,)
    if with_em: r.update(closed_em=em(m, CPq, bans), evidence_em=em(m, EPq, bans))
    if critic:
        r.update(isrel_g2=auroc(G2W.g2_label, critique_scores(m, G2W.prompt.tolist(), DEV)),
                 isrel_val=auroc(EVW.auto_rel, critique_scores(m, EVW.prompt.tolist(), DEV)))
    return r
sd = lambda p: torch.load(p, map_location="cpu", weights_only=False)
st_route = sd(ROUTER); SD_ROUTE = st_route["model"]
work = load_model(ROUTER, DEV)
INT = []
for s in SEEDS:
    SD_SFT = sd(E1[s][-1][1])["model"]
    for a in ALPHAS:
        work.load_state_dict(interpolate(SD_ROUTE, SD_SFT, a))
        INT.append({"kind": "interp", "seed": s, "alpha": a, **measure(work)}); r = INT[-1]
        stage(f"seed {s} alpha {a:.1f}: routing AUROC {r['route_auroc']:.3f} | call {r['default_call']:.1%} | instr loss {r['instr_val_loss']:.3f} | closed {r['closed_em']:.1%} | evidence {r['evidence_em']:.1%}")
SD_T0 = sd(THETA0)["model"]
for s in SEEDS:
    SD_SFT = sd(E1[s][-1][1])["model"]
    for b in BETAS:
        work.load_state_dict(task_vector_add(SD_SFT, SD_ROUTE, SD_T0, b))
        INT.append({"kind": "task_vector", "seed": s, "alpha": None, "beta": b, **measure(work)}); r = INT[-1]
        stage(f"seed {s} task vector beta {b}: routing AUROC {r['route_auroc']:.3f} | instr loss {r['instr_val_loss']:.3f} | closed {r['closed_em']:.1%}")
del work
INT = pd.DataFrame(INT)
L0 = INT[(INT.kind == "interp") & (INT.alpha == 0.0)].instr_val_loss.mean()
INT["sft_gain_kept"] = [(L0 - l) / (L0 - INT[(INT.kind == "interp") & (INT.seed == s) & (INT.alpha == 1.0)].instr_val_loss.item())
                        for l, s in zip(INT.instr_val_loss, INT.seed)]
INT.to_csv(f"{OUT}/interpolation.csv", index=False)
display(INT.round(4))
A0 = INT[(INT.kind == "interp") & (INT.alpha == 0.0)].route_auroc.mean()
SUMM2 = []
for s in SEEDS:
    d = INT[(INT.kind == "interp") & (INT.seed == s)]
    ok = d[d.route_auroc >= A0 - 0.02]; a_star = ok.alpha.max()
    g90 = d[d.sft_gain_kept >= 0.9]; a90 = g90.alpha.min()
    SUMM2.append({"seed": s, "alpha_star(route>=E0-0.02)": a_star, "gain_kept_at_alpha_star": d[d.alpha == a_star].sft_gain_kept.item(),
                  "alpha_min(gain>=0.9)": a90, "route_auroc_at_that_alpha": d[d.alpha == a90].route_auroc.item()})
SUMM2 = pd.DataFrame(SUMM2); display(SUMM2.round(4))

work = load_model(CRITIC, DEV); SD_C0 = sd(CRITIC)["model"]; CINT = []
for s in SEEDS:
    SD_C1 = sd(C1[s])["model"]
    for a in C_ALPHAS:
        work.load_state_dict(interpolate(SD_C0, SD_C1, a))
        CINT.append({"seed": s, "alpha": a, **measure(work, with_em=False, critic=True)}); r = CINT[-1]
        stage(f"critic seed {s} alpha {a:.1f}: IsRel G2 {r['isrel_g2']:.3f} | IsRel val {r['isrel_val']:.3f} | routing {r['route_auroc']:.3f} | instr loss {r['instr_val_loss']:.3f}")
del work
CINT = pd.DataFrame(CINT); CINT.to_csv(f"{OUT}/interpolation_critic.csv", index=False); display(CINT.round(4))

## 6 · Analysis 6 — threshold recalibration (and the re-fitted probe with the same labels)

In [ ]:
emA, emC = V.em_A.values, V.em_C.values
D0 = SC[("E0", "all", 0.0)]["token"] > 0
def evaluate_decision(d):
    return {"balanced_acc": balanced_accuracy(d, yV), "call_rate": float(d.mean()), "agree_E0": float((d == D0).mean()),
            "U_theta0": float(np.where(d, emC, emA).mean() - LAM * d.mean())}
rng = np.random.default_rng(SEED); CAL = []
for key in [("E0", "all", 0.0)] + [("E1", s, 1.0) for s in SEEDS]:
    hb, hv, sv = F[key]
    mk_ = load_model(ROUTER if key[0] == "E0" else E1[key[1]][-1][1], DEV); sB = router_scores(mk_, RP_B, "action", DEV)[0]; del mk_
    CAL.append({"model": f"{key[0]}_s{key[1]}", "method": "default threshold 0", "n_labels": 0, **evaluate_decision(sv > 0)})
    CAL.append({"model": f"{key[0]}_s{key[1]}", "method": "oracle threshold (val, upper bound)", "n_labels": len(yV), **evaluate_decision(sv > fit_threshold(sv, yV))})
    for n in [min(n, len(yB)) for n in CAL_NS]:
        for rep in range(REPS):
            idx = rng.choice(len(yB), n, replace=False)
            if not 0 < yB[idx].sum() < n: continue
            t = fit_threshold(sB[idx], yB[idx])
            CAL.append({"model": f"{key[0]}_s{key[1]}", "method": "recalibrated threshold", "n_labels": n, "rep": rep, **evaluate_decision(sv > t)})
            pr = fit_probe(hb[idx], yB[idx]); tp = fit_threshold(probe_scores(pr, hb[idx]), yB[idx])
            CAL.append({"model": f"{key[0]}_s{key[1]}", "method": "re-fitted probe + threshold", "n_labels": n, "rep": rep,
                        **evaluate_decision(probe_scores(pr, hv) > tp), "auroc": auroc(yV, probe_scores(pr, hv))})
CAL = pd.DataFrame(CAL); CAL.to_csv(f"{OUT}/recalibration.csv", index=False)
display(CAL.groupby(["model", "method", "n_labels"])[["balanced_acc", "call_rate", "agree_E0", "U_theta0"]].mean().round(4))

## 7 · Save + checks

In [ ]:
R1.to_csv(f"{OUT}/readout_routing.csv", index=False); R2.to_csv(f"{OUT}/readout_isrel.csv", index=False); LC.to_csv(f"{OUT}/probe_learning_curve.csv", index=False)
SUMM2.to_csv(f"{OUT}/interpolation_summary.csv", index=False)
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("E0_token_is_06_router", abs(tok_pre - auroc(yV, V.router_score)) < 1e-3, f"E0 token AUROC {tok_pre:.4f}")
p07 = pd.read_csv(os.path.join(D07, "pilot_table.csv")); p07 = p07[p07.frac == 1.0]
ref07 = {int(r.seed): r.auroc for r in p07.itertuples()}
e1 = {s: R1[(R1.seed == s) & (R1.frac == 1.0)].token.item() for s in SEEDS}
add("E1_token_matches_07", all(abs(e1[s] - ref07[s]) < 1e-3 for s in SEEDS), f"{e1} vs 07 {ref07}")
i0 = INT[(INT.kind == "interp") & (INT.alpha == 0.0)].route_auroc; i1 = {s: INT[(INT.kind == "interp") & (INT.seed == s) & (INT.alpha == 1.0)].route_auroc.item() for s in SEEDS}
add("interpolation_endpoints", (abs(i0 - tok_pre) < 2e-3).all() and all(abs(i1[s] - e1[s]) < 2e-3 for s in SEEDS), f"alpha 0 {list(i0.round(4))}, alpha 1 {i1}")
c0 = R2[(R2.model == "critic") & (R2["eval"] == "G2")].token.item()
add("critic_matches_10", abs(c0 - 0.8719) < 2e-3, f"critic IsRel G2 {c0:.4f} (notebook 10: 0.8719)", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"notebook": "11_posthoc", "status": status, "checks": checks, "recovery": REC.to_dict("records"), "interpolation_summary": SUMM2.to_dict("records"),
           "code": {"repo": REPO_URL, "ref": REPO_REF, "commit": CODE_COMMIT}}, open(f"{OUT}/report.json", "w"), indent=1, default=float)
print(f"\nNOTEBOOK 11: {status}")